# 2. Profile Dataset

Profile a logged immutable dataset from MLflow/GCS artifacts. This notebook does not rebuild the data pipeline, refresh source data, or derive a train/test split.

## 1. Load Project

In [ ]:
from __future__ import annotations

import pandas as pd
from IPython.display import display

import automl
from automl import data

DRY_RUN = True


In [ ]:
active = automl.use_project(dry_run=DRY_RUN)
config = active.config
display(
    {
        "project": active.project_name,
        "repo_root": str(config.repo_root),
        "project_dir": str(config.project_dir),
        "experiment": active.active_experiment_id,
        "dry_run": active.dry_run,
    }
)


## 2. List Logged Datasets

In [ ]:
index = data.list_datasets(session=active)
index.to_dataframe()


## 3. Select Dataset Explicitly

The default is the active dataset. Edit `DATASET_ID` if you want a different logged dataset.

In [ ]:
DATASET_ID = index.active_dataset_id or (index.datasets[-1].id if index.datasets else None)
DATASET_ID


## 4. Load Dataset From Artifacts

This reads the full dataset dataframe, feature registry, and manifest from logged artifacts. It does not call the data builder.

In [ ]:
if DATASET_ID is None:
    raise RuntimeError("No logged dataset is available. Run materialization first.")

loaded_dataset = data.load_dataset_by_id(DATASET_ID, session=active)
loaded_dataset.dataset.id


## 5. Dataset Sanity Views

In [ ]:
loaded_dataset.df.head()


In [ ]:
loaded_dataset.df.shape


In [ ]:
loaded_dataset.registry.to_dataframe()


In [ ]:
loaded_dataset.dataset.to_dict()


In [ ]:
target_col = loaded_dataset.dataset.target_column
loaded_dataset.df[target_col].value_counts(dropna=False).to_frame("rows")


In [ ]:
loaded_dataset.df.isna().mean().sort_values(ascending=False).to_frame("null_rate").head(30)


In [ ]:
loaded_dataset.df.dtypes.astype(str).value_counts().to_frame("columns")


In [ ]:
categorical_cols = loaded_dataset.df.select_dtypes(include=["object", "category"]).columns
loaded_dataset.df[categorical_cols].nunique(dropna=True).sort_values(ascending=False).to_frame("cardinality").head(30)


In [ ]:
loaded_dataset.registry.to_dataframe()[["available", "feature", "target", "model"]].sum().to_frame("columns")


In [ ]:
{
    "source_identity": loaded_dataset.dataset.source_identity,
    "component_hashes": loaded_dataset.dataset.component_hashes.to_dict(),
}


## 6. Ad Hoc Exploration Beyond The Standard Profile

These cells display investigation tables for a human reviewer. They do not mutate `config.py`.

In [ ]:
unique_key = list(loaded_dataset.dataset.unique_key)
unique_key_health = pd.Series(
    {
        "unique_key": unique_key,
        "duplicate_key_rows": int(loaded_dataset.df.duplicated(subset=unique_key, keep=False).sum()) if unique_key else None,
        "null_key_rows": int(loaded_dataset.df[unique_key].isna().any(axis=1).sum()) if unique_key else None,
    }
)
unique_key_health


In [ ]:
loaded_dataset.df.loc[loaded_dataset.df.duplicated(subset=unique_key, keep=False), unique_key].head(20) if unique_key else pd.DataFrame()


In [ ]:
full_row_duplication = pd.Series(
    {
        "duplicate_rows": int(loaded_dataset.df.duplicated(keep=False).sum()),
        "duplicate_rate": float(loaded_dataset.df.duplicated(keep=False).mean()),
    }
)
full_row_duplication


In [ ]:
split_pct_col = loaded_dataset.dataset.split_pct_col
(
    loaded_dataset.df[split_pct_col]
    .value_counts(dropna=False)
    .sort_index()
    .reindex(range(100), fill_value=0)
    .to_frame("rows")
    if split_pct_col in loaded_dataset.df.columns
    else pd.DataFrame()
)


In [ ]:
target_missing = loaded_dataset.df[target_col].isna()
{
    "missing_target_rows": int(target_missing.sum()),
    "examples": loaded_dataset.df.loc[target_missing].head(10),
}


In [ ]:
missing_by_target = []
if loaded_dataset.df[target_col].nunique(dropna=True) <= 20:
    for col in loaded_dataset.df.columns:
        if col == target_col:
            continue
        rates = loaded_dataset.df.assign(_missing=loaded_dataset.df[col].isna()).groupby(target_col, dropna=False)["_missing"].mean()
        if len(rates) > 1 and rates.max() - rates.min() >= 0.25:
            missing_by_target.append({"column": col, "min_null_rate": rates.min(), "max_null_rate": rates.max()})
pd.DataFrame(missing_by_target).sort_values("max_null_rate", ascending=False) if missing_by_target else pd.DataFrame()


In [ ]:
rare_categories = []
for col in categorical_cols:
    counts = loaded_dataset.df[col].value_counts(dropna=False)
    tiny = counts[counts <= max(2, len(loaded_dataset.df) * 0.005)]
    if len(tiny):
        rare_categories.append({"column": col, "rare_category_count": len(tiny), "examples": tiny.head(5).to_dict()})
pd.DataFrame(rare_categories).sort_values("rare_category_count", ascending=False) if rare_categories else pd.DataFrame()


In [ ]:
high_cardinality = (
    loaded_dataset.df[categorical_cols]
    .nunique(dropna=True)
    .sort_values(ascending=False)
    .to_frame("cardinality")
)
high_cardinality[high_cardinality["cardinality"] > 50]


In [ ]:
near_constant = []
for col in loaded_dataset.df.columns:
    counts = loaded_dataset.df[col].value_counts(dropna=False, normalize=True)
    if len(counts) <= 1 or (len(counts) and counts.iloc[0] >= 0.995):
        near_constant.append({"column": col, "dominant_share": float(counts.iloc[0]) if len(counts) else 0.0})
pd.DataFrame(near_constant).sort_values("dominant_share", ascending=False) if near_constant else pd.DataFrame()


In [ ]:
datetime_review = []
for col in loaded_dataset.df.columns:
    if "DATE" in col.upper() or "TIME" in col.upper():
        parsed = pd.to_datetime(loaded_dataset.df[col], errors="coerce")
        datetime_review.append(
            {
                "column": col,
                "parse_failure_rate": float(parsed.isna().mean()),
                "min": parsed.min(),
                "max": parsed.max(),
                "future_rows": int((parsed > pd.Timestamp.utcnow().tz_localize(None)).sum()) if parsed.notna().any() else 0,
            }
        )
pd.DataFrame(datetime_review) if datetime_review else pd.DataFrame()


In [ ]:
numeric = loaded_dataset.df.select_dtypes(include="number")
outlier_scan = numeric.quantile([0, 0.001, 0.01, 0.5, 0.99, 0.999, 1.0]).T
outlier_scan.head(50)


In [ ]:
leakage_terms = ("TARGET", "LABEL", "OUTCOME", "STATUS", "DEFAULT", "PAID", "APPROVED")
name_based_suspects = [
    col for col in loaded_dataset.df.columns
    if col != target_col and any(term in col.upper() for term in leakage_terms)
]
name_based_suspects


In [ ]:
registry_df = loaded_dataset.registry.to_dataframe()
registry_columns = set(registry_df["name"])
data_columns = set(loaded_dataset.df.columns)
registry_mismatches = {
    "data_missing_from_registry": sorted(data_columns - registry_columns),
    "registry_missing_from_data": sorted(registry_columns - data_columns),
    "target_rows": registry_df[registry_df["target"] == True],
    "metadata_or_excluded_target_conflicts": registry_df[(registry_df["target"] == True) & (registry_df["feature"] == True)],
}
registry_mismatches


## 7. Run Deterministic Profile

This writes local profile artifacts and logs them to the experiment overview run using `MlflowClient` with an explicit run id.

In [ ]:
profile_result = data.profile(dataset_id=DATASET_ID, session=active)
pd.DataFrame([profile_result.to_dict()])


## 8. MLflow Artifact Logging Contract

Expected logged artifacts:

- `{DATASET_ID}/profile/data_card.json`
- `{DATASET_ID}/profile/data_observations.json`
- `{DATASET_ID}/profile/profile_manifest.json`
- `{DATASET_ID}/profile/charts/...`

In [ ]:
{
    "data_card_uri": profile_result.data_card_uri,
    "data_observations_uri": profile_result.data_observations_uri,
    "profile_manifest_uri": profile_result.profile_manifest_uri,
    "chart_uris": profile_result.chart_uris,
}


## 9. Inspect Logged Profile Artifacts

In [ ]:
pd.DataFrame([profile_result.to_dict()])
